# **ML : Practical - 10**

Mohd Talha Patrawala

CMPN - B

23102B0025

In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


In [2]:
categories = ['sci.space', 'rec.autos', 'talk.politics.misc']

data = fetch_20newsgroups(subset='all',
                         categories=categories,
                         remove=('headers', 'footers', 'quotes'))

docs = data.data

print("Docs:", len(docs))

Docs: 2752


In [3]:
data

{'data': ["\n\n\n\n\n\nYour bad English?  (See quote above.)\n\n\n\nYou'd lose that wager, if the supporting argument were part of it.\n\n\nDid you know that Hitler himself was a devout Christian?  And heterosexual?\n\n--Drywid",
  "I was a graduate student in the early 1980s, and we had a conference on \nReaganomics where Jerry Jordan, then a member of the Council of Economic \nAdvisors, was a speaker.  I had the pleasure of driving him back to the \nairport afterwards, and since taxes were the main topic of discussion I \nthought I would ask him about the VAT.  I have favored it for these reasons \nyou mention, that the income base is too hazy to define, that it taxes \nsavings and investment, that it is likely to be more visible.  He agreed, \nand reported that the CEA at that time was in favor of VAT.  So why not \npropose it?  I asked.  He replied that the Reagan White House feared that \nthe Democrats would introduce VAT *in addition to* the income tax, rather \nthan in lieu.  Be

In [4]:
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)

X = tfidf.fit_transform(docs)

print("Shape:", X.shape)


Shape: (2752, 5000)


In [5]:
components_list = [50, 100, 200]

results = []

terms = tfidf.get_feature_names_out()


In [6]:
file = open("lsa_topic_terms.txt", "w")

for n in components_list:

    print("\n--- Components:", n, "---")

    # SVD
    svd = TruncatedSVD(n_components=n, random_state=42)
    X_svd = svd.fit_transform(X)

    var = svd.explained_variance_ratio_.sum()
    print("Explained variance:", var)

    # KMeans clustering
    km = KMeans(n_clusters=3, random_state=42)
    labels = km.fit_predict(X_svd)

    score = silhouette_score(X_svd, labels)
    print("Silhouette score:", score)

    results.append([n, var, score])

    # Topics (top words)
    file.write("\nComponents: " + str(n) + "\n")

    for i in range(5):   # first 5 topics
        comp = svd.components_[i]

        top_idx = comp.argsort()[-10:][::-1]

        words = []
        for j in top_idx:
            words.append(terms[j])

        line = "Topic " + str(i) + ": " + ", ".join(words)

        print(line)
        file.write(line + "\n")


file.close()



--- Components: 50 ---
Explained variance: 0.123297763331
Silhouette score: 0.05338898618022494
Topic 0: car, people, like, just, space, don, think, know, good, new
Topic 1: car, cars, engine, dealer, ford, oil, good, miles, price, drive
Topic 2: space, car, shuttle, nasa, launch, orbit, mission, engine, thanks, satellite
Topic 3: men, homosexual, gay, sex, sexual, homosexuals, partners, male, cramer, promiscuous
Topic 4: insurance, health, car, private, space, thanks, tax, government, mail, care

--- Components: 100 ---
Explained variance: 0.19745980300066118
Silhouette score: 0.05122531679167978
Topic 0: car, people, like, just, space, don, think, know, good, new
Topic 1: car, cars, engine, dealer, ford, oil, good, miles, price, drive
Topic 2: space, car, shuttle, nasa, launch, orbit, mission, engine, thanks, satellite
Topic 3: men, homosexual, gay, sex, sexual, homosexuals, partners, male, cramer, promiscuous
Topic 4: insurance, health, car, private, space, thanks, tax, government,

In [7]:
df = pd.DataFrame(results, columns=["components", "variance", "silhouette"])
df.to_csv("svd_results.csv", index=False)

print("\nSaved files successfully")

df


Saved files successfully


,components,variance,silhouette
0,50,0.123298,0.053389
1,100,0.197460,0.051225
2,200,0.314065,0.041627


# **Conclusion:**

Singular Value Decomposition (SVD) reduces high-dimensional, sparse text matrices by decomposing them into lower-rank representations that capture the most important patterns (latent semantics). This removes noise and redundancy while preserving meaningful relationships between terms and documents, making computation more efficient and improving performance in tasks like search and classification.
